<h1 style="text-align: center;">Telkomsel Exploratory Data Analysis</h1>
<h3 style="text-align: center;">Muhammad Rafi Andrianto</h3>
<h3 style="text-align: center;">Yoanita Dwi Harlandi</h3>

---

## **Section 1. Business Context**

**1.1 Context**

Sebagai operator telekomunikasi terbesar di Indonesia yang mengelola infrastruktur broadband berskala masif, Telkomsel menghadapi perubahan lanskap industri yang dinamis. Pergeseran perilaku konsumen dari layanan legacy (Voice/SMS) menuju layanan Data/Internet memaksa perusahaan terus berinovasi menelurkan berbagai produk kuota data guna mengakuisisi pengguna prabayar baru secara masif melalui aplikasi MyTelkomsel maupun jaringan konter tradisional (Digipos).

Namun, persaingan harga yang sangat agresif (price war) di pasar seluler Indonesia melahirkan tantangan internal yang serius:

1. Fenomena **"Bakar Perdana"**: Paket kuota promo pada kartu perdana baru sengaja di-subsidi agar harganya jauh lebih murah per Gigabyte dibandingkan biaya perpanjangan (top-up) paket normal. Alih-alih melakukan loyalitas pengisian ulang, banyak konsumen memilih membuang kartu SIM lama mereka setelah kuotanya habis dan membeli kartu perdana baru. Hal ini menciptakan metrik semu di mana pertumbuhan pelanggan baru terlihat sangat tinggi (Gross Additions), namun diiringi oleh tingkat kerontokan pelanggan yang luar biasa besar (Churn Rate). Akibatnya, nilai rata-rata pendapatan per pengguna atau ARPU (Average Revenue Per User) perusahaan mengalami penurunan tajam.

2. **Abuse Program Device Bundling**: Kebocoran anggaran subsidi juga terdeteksi pada program kerjasama perangkat premium (misal: beli smartphone gratis kuota 1 tahun). Terdapat indikasi sindikat fraud yang membeli perangkat bundling tersebut untuk diambil kartu SIM subsidi kuotanya demi dijual eceran (dimanfaatkan pada perangkat modem atau HP lain), sedangkan unit smartphone-nya dijual terpisah. Hal ini membuat anggaran subsidi bernilai miliaran rupiah melayang tanpa menghasilkan pelanggan bernilai tinggi (high-value subscribers) yang diharapkan.

Melalui pendekatan berbasis data, tim Data Analytics dituntut untuk melakukan analisis forensik terhadap log aktivitas penggunaan data (Call Data Record - CDR), pelacakan nomor identitas perangkat (IMEI), dan status kepatuhan registrasi prabayar. Langkah ini mendesak dilakukan agar Telkomsel dapat mengubah strategi bisnis dari yang semula berfokus pada kuantitas (mengejar jumlah pengguna baru) menjadi berfokus pada kualitas (retensi pelanggan setia yang profitabel).

**1.2 Problem Statements**

Masalah utama yang dihadapi perusahaan adalah merosotnya nilai ARPU (Average Revenue Per User) serta terjadinya kebocoran anggaran subsidi akibat praktik fraud "Bakar Perdana" dan manipulasi program Device Bundling (IMEI Mismatch).

Secara operasional dan teknis data, masalah tersebut dirumuskan ke dalam beberapa pertanyaan turunan berikut:
1. **Analisis Churn & Regional**: Di area atau regional mana tingkat peredaran kartu "Bakar Perdana" (terindikasi dari tingginya angka churn di bulan pertama) berada pada level tertinggi?

2. **Estimasi Kebocoran Anggaran**: Berapa besar kerugian finansial yang dialami perusahaan akibat penyalahgunaan paket bundling premium di mana identitas perangkat riil yang digunakan berbeda dengan perangkat yang didaftarkan (registered_imei $\neq$ used_imei)?

3. **Standarisasi Infrastruktur Data**: Seberapa besar tingkat inkonsistensi penulisan tipe jaringan (network_type) pada log BTS yang masuk ke database pusat, yang berpotensi memicu bias pada visualisasi performa jaringan broadband?

4. **Risiko Kepatuhan Regulasi (Compliance Risk)**: Berapa persentase nomor pelanggan yang tidak memiliki kejelasan status registrasi NIK/KK, yang menempatkan perusahaan pada risiko denda audit dari Kominfo?

5. **Anomali Sistem & Sinkronisasi**: Berapa banyak sesi penggunaan internet yang bocor secara ilegal sebelum kartu SIM resmi diaktivasi (session_time < activation_date) akibat ketidaksinkronan sistem HLR dan Billing?
    - Berapa banyak temuan outlier pemakaian data ekstrem (nilai fiktif bug billing) yang dapat merusak analisis rata-rata konsumsi data riil pelanggan?


**1.3 Key Objective**

Tujuan utama dari projek analisis ini adalah memberikan rekomendasi strategis berbasis data untuk menghentikan kebocoran pendapatan perusahaan, dengan sasaran spesifik sebagai berikut:

- Mitigasi Churn & Fraud: Mengidentifikasi pola dan wilayah penyebaran kartu "Bakar Perdana" guna merancang kebijakan pembatasan distribusi atau blacklisting ID Outlet konter yang memfasilitasi registrasi massal ilegal.

- Perlindungan Subsidi (Subsidy Protection): Menyediakan basis data untuk penerapan sistem IMEI Locking, sehingga kuota subsidi otomatis hangus jika kartu SIM dipindahkan ke perangkat non-mitra.

- Kepatuhan Sistem (Regulatory Compliance): Mengisolasi nomor-nomor dengan status NIK/KK kosong agar segera dilakukan pemblokiran massal guna mematuhi regulasi Kominfo dan menghindari denda eksternal.

- Integritas Data & Perbaikan Sistem: Menyediakan laporan bug (System Bug Report) bagi tim Data Engineering dan Network Operations untuk menutup celah kebocoran kuota pada database HLR serta membersihkan anomali pencatatan sistem billing.

## **Section 2. Data Understanding**

**2.1 General Information**

In [1]:
import pandas as pd
import numpy as np

In [2]:
sub_raw = pd.read_csv('../data/raw/tsel_subscribers.csv')
pkg_raw = pd.read_csv('../data/raw/tsel_packages.csv')
usage_raw = pd.read_csv('../data/raw/tsel_data_usage.csv')

print(f"Subscribers Shape : {sub_raw.shape}")
print(f"Packages Shape    : {pkg_raw.shape}")
print(f"Data Usage Shape  : {usage_raw.shape}")

Subscribers Shape : (35000, 4)
Packages Shape    : (10, 3)
Data Usage Shape  : (300000, 7)


- tsel_subscribers.csv: Terdiri dari 35.000 baris dan 4 kolom (Master data profil pelanggan)
- tsel_packages.csv: Terdiri dari 10 baris dan 3 kolom (Katalog produk paket data)
- tsel_data_usage.csv: Terdiri dari 300.000 baris dan 7 kolom (Log aktivitas Call Data Record / CDR)

**2.2 Feature Information**
 
**2.2.1 tsel_subscribers.csv**

| Nama Kolom | Deskripsi |
| :--- | :--- |
| **msisdn** | Nomor HP pelanggan (MSISDN). |
| **activation_date** | Tanggal kartu SIM diaktifkan/registrasi. |
| **nik_kk_status** | Status registrasi NIK/KK. |
| **registered_imei** | IMEI smartphone yang didaftarkan saat beli paket bundling (Bisa NaN jika bukan paket bundling). |


**2.2.2 tsel_packages.csv**


| Nama Kolom | Deskripsi |
| :--- | :--- |
| **package_code** | Kode unik paket. |
| **package_name** | Nama paket komersial (Promo Perdana, Halo Bundling, OMG!, Kuota Ketengan). |
| **price** | Harga paket. |

**2.2.3 tsel_data_usage.csv**

| Nama Kolom | Deskripsi |
| :--- | :--- |
| **session_id** | ID unik sesi internet. |
| **msisdn** | Nomor HP pengguna. |
| **package_code** | Kode paket yang memotong kuota. |
| **used_imei** | IMEI HP yang real-time dipakai saat internetan (Jika berbeda dengan registered_imei, berarti Abuse Bundling). |
| **session_time** | Tanggal & Jam pengguna menyalakan koneksi data. |
| **network_type** | Jaringan yang dipakai. |
| **payload_mb** | Jumlah Megabyte (MB) yang disedot dalam sesi tersebut. |
| **(Business Context Error)** | Sesi penggunaan internet di session_time terjadi sebelum activation_date dari kartu SIM tersebut. (Logika fiktif: Kartu belum didaftarin NIK tapi udah bisa buat streaming YouTube). |



In [3]:
print(f"--- subscribers data type ---\n{sub_raw.dtypes}\n")
print(f"--- package data type ---\n{pkg_raw.dtypes}\n")
print(f"--- usage data type ---\n{usage_raw.dtypes}")

--- subscribers data type ---
msisdn               int64
activation_date        str
nik_kk_status          str
registered_imei    float64
dtype: object

--- package data type ---
package_code      str
package_name      str
price           int64
dtype: object

--- usage data type ---
session_id          str
msisdn            int64
package_code        str
used_imei         int64
session_time        str
network_type        str
payload_mb      float64
dtype: object


**2.3 Statistics Summary**

In [4]:
usage_raw['payload_mb'].describe()

count    300000.000000
mean       6159.144994
std       71485.176443
min        -150.500000
25%         506.337500
50%        1024.020000
75%        1541.155000
max      999999.900000
Name: payload_mb, dtype: float64

Evaluasi metrik statistik deskriptif pada kolom inti payload_mb menunjukkan anomali matematika:

- Penyimpangan Rata2 (Skewness Ekstrem): Nilai rata2 (mean) penggunaan internet berada di angka 6.159,14 MB, sangat jauh melompati nilai tengahnya (median/50%) yang hanya berada di angka 1.024,02 MB. Hal ini membuktikan adanya tarikan dari nilai outliers yang sangat extreme.

- Indikasi Error Sistem: Nilai minimum berada pada angka -150.50 MB, sebuah nilai negatif yang tidak masuk akal untuk kuota pemakaian data fisik

- Nilai maksimum berada pada angka 999.999,90 MB (~1 Terabyte), nilai yang di luar normal

## **Section 3. Data Cleaning**

Tahap ini untuk mengidentifikasi cacat log data seperti anomali, nilai kosong, inkonsistensi format, dan variabel duplikat. Langkah ini menjamin bahwa pengolahan data pada tahap berikutnya menghasilkan visualisasi dan metrik2 yang akurat tanpa bias akibat kerusakan data/sistem

**3.1 Missing Values**

In [5]:
print("--- tsel_subscribers ---")
print(sub_raw.isna().sum())

print("\n--- tsel_data_usage ---")
print(usage_raw.isna().sum())

invalid_nik = sub_raw[sub_raw['nik_kk_status'].isna()]
pct_invalid_nik = (len(invalid_nik) / len(sub_raw)) * 100

print(f"\nPersentase NIK/KK Status bermasalah/missing: {pct_invalid_nik:.2f}%")

--- tsel_subscribers ---
msisdn                 0
activation_date        0
nik_kk_status       5202
registered_imei    24470
dtype: int64

--- tsel_data_usage ---
session_id      0
msisdn          0
package_code    0
used_imei       0
session_time    0
network_type    0
payload_mb      0
dtype: int64

Persentase NIK/KK Status bermasalah/missing: 14.86%


- **nik_kk_status (5202)**: Dari perspektif hukum dan kepatuhan (regulatory compliance), ketiadaan status registrasi sebesar 14,86% merupakan sinyal bahaya risiko denda audit dari Kominfo. Baris ini tidak boleh dihapus dari basis data historis karena nomor-nomor ini harus diisolasi untuk dilaporkan ke tim Fraud & Security guna dilakukan pemblokiran massal.

- **registered_imei (24470)**: Berdasarkan analisis konteks bisnis, ketiadaan data IMEI terdaftar ini bersifat normal dan valid. Pelanggan prabayar reguler yang membeli kartu perdana biasa (non-bundling smartphone) memang tidak meregistrasikan IMEI perangkat mereka saat aktivasi. Mereka membeli kartu SIM secara terpisah untuk dimasukkan ke HP apa saja yang sudah mereka miliki 

**3.2 Duplicated Values**

In [6]:
print(f"Duplikat di tsel_subscribers : {sub_raw.duplicated().sum()}")
print(f"Duplikat di tsel_packages    : {pkg_raw.duplicated().sum()}")
print(f"Duplikat di tsel_data_usage  : {usage_raw.duplicated().sum()}")

Duplikat di tsel_subscribers : 0
Duplikat di tsel_packages    : 0
Duplikat di tsel_data_usage  : 0


Hasil pengujian mengonfirmasi tidak ada record yang terduplikasi di seluruh baris tabel. Hal ini membuktikan bahwa struktur keunikan data Primary Key (msisdn, package_code, session_id) berada dalam kondisi aman dan terbebas dari redundansi sistem pencatatan log.

**3.3 Identify Spelling Errors**

In [7]:
usage_raw['network_type'].value_counts()

network_type
4G        89981
LTE       44884
4G LTE    30339
5G        30075
4g        29833
WCDMA     15225
3G        15076
NR        14941
HSDPA     14905
5g        14741
Name: count, dtype: int64

bisa dilihat adanya inkonsistensi penulisan teks yang terjadi akibat variasi format pengiriman log teknis dari mesin vendor tower BTS yang berbeda2 ke database pusat. Jika inkonsistensi ini tidak ditangani, hasil agregasi performa jaringan broadband akan pecah menjadi 10 bagian terpisah (misalnya performa 4G seolah berbeda dengan LTE atau 4g), sehingga visualisasi data menjadi bias dan dapat menyesatkan stakeholders

**3.4 Identify Anomaly Values**

In [8]:
sub_dt = sub_raw[['msisdn', 'activation_date']].copy()
sub_dt['activation_date'] = pd.to_datetime(sub_dt['activation_date'])

usage_dt = usage_raw[['msisdn', 'session_time', 'payload_mb', 'session_id']].copy()
usage_dt['session_time'] = pd.to_datetime(usage_dt['session_time'])

#merge untuk validasi tanggal aktivasi dengan tanggal session penggunaan
df_merge_time = pd.merge(usage_dt, sub_dt, on='msisdn', how='left')
anomaly_time = df_merge_time[df_merge_time['session_time'] < df_merge_time['activation_date']]

print(f"jumlah sesi internet SEBELUM tanggal aktivasi: {len(anomaly_time)}")
anomaly_time.head()

jumlah sesi internet SEBELUM tanggal aktivasi: 12008


,msisdn,session_time,payload_mb,session_id,activation_date
26,82254454198,2023-04-18,133.47,CDR000000027,2023-04-20
34,82222192600,2023-10-04,758.03,CDR000000035,2023-10-11
63,85272766900,2023-04-07,1090.95,CDR000000064,2023-04-09
104,82120468327,2023-10-17,760.71,CDR000000105,2023-10-18
124,81188580169,2023-02-15,409.70,CDR000000125,2023-02-18


Secara logika bisnis prabayar, aktivitas internetan sebelum registrasi kartu SIM adalah kegagalan sistem yang mustahil (kebocoran kuota gratis). Hal ini mengonfirmasi adanya bug sinkronisasi antara database dengan sistem penagihan.

In [9]:
outlier_extreme = usage_raw[usage_raw['payload_mb'] >= 50000] # max sementara 50 GB per sesi
outlier_extreme_min = usage_raw[usage_raw['payload_mb'] < 0] 
print(f"\njumlah sesi yang melebihi 50,000 MB: {len(outlier_extreme)}")
print(f"jumlah sesi yang dibawah dari 0 MB (negatif): {len(outlier_extreme_min)}\n")

print("top nilai terbesar payload_mb:")
print(usage_raw['payload_mb'].nlargest())

print("\ntop nilai negatif payload_mb:")
print(usage_raw['payload_mb'].nsmallest())


jumlah sesi yang melebihi 50,000 MB: 1544
jumlah sesi yang dibawah dari 0 MB (negatif): 1469

top nilai terbesar payload_mb:
2       999999.9
451     999999.9
1118    999999.9
1353    999999.9
1529    999999.9
Name: payload_mb, dtype: float64

top nilai negatif payload_mb:
52    -150.5
144   -150.5
270   -150.5
280   -150.5
300   -150.5
Name: payload_mb, dtype: float64


nilai 999.999,9 MB (~1 Terabyte dalam satu sesi) muncul sebanyak 1.544 kali dan nilai -150,5 MB muncul sebanyak 1469 kali. Jika dipertahankan, nilai korup ini mendistorsi nilai rata2 (mean) penggunaan data seperti yang dari seharusnya ~1 GB melonjak ke 6,1 GB, sehingga merusak pemodelan kapasitas jaringan.

**3.5 Creating Clean Datasets**

In [10]:
sub_clean = sub_raw.copy()
pkg_clean = pkg_raw.copy()
usage_clean = usage_raw.copy()

sub_clean['activation_date'] = pd.to_datetime(sub_clean['activation_date'])
usage_clean['session_time'] = pd.to_datetime(usage_clean['session_time'])

In [11]:
#Tangani Missing Value pada NIK/KK Status (Ubah NaN jadi missing/unregis)
sub_clean['nik_kk_status'] = sub_clean['nik_kk_status'].fillna('Missing/Unregistered')

In [12]:
#Standardisasi Network Type (mapping 10 variasi menjadi 3)
network_map = {
    '4G': '4G', 
    'LTE': '4G', 
    '4G LTE': '4G', 
    '4g': '4G',
    '5G': '5G', 
    'NR': '5G', 
    '5g': '5G',
    'WCDMA': '3G', 
    '3G': '3G', 
    'HSDPA': '3G'
}
usage_clean['network_type'] = usage_clean['network_type'].map(network_map)
usage_clean['network_type'].value_counts()

network_type
4G    195037
5G     59757
3G     45206
Name: count, dtype: int64

In [13]:
#Hapus outlier dan value aneh pada Payload MB (hapus nilai < 0 dan nilai extreme 999999.9)
usage_clean = usage_clean[(usage_clean['payload_mb'] >= 0) & (usage_clean['payload_mb'] < 999999.9)]

#Hapus sesi internet yang dipakai sebelum tanggal aktivasi kartu
#merge sementara dengan data activation_date untuk memfilter baris yang valid
df_merge_filter = pd.merge(usage_clean, sub_clean[['msisdn', 'activation_date']], on='msisdn', how='left')
usage_clean = df_merge_filter[df_merge_filter['session_time'] >= df_merge_filter['activation_date']].drop(columns=['activation_date'])

In [14]:
usage_clean['payload_mb'].describe()

count    285123.000000
mean       1023.542969
std         590.941434
min           0.010000
25%         511.490000
50%        1023.800000
75%        1535.645000
max        2047.980000
Name: payload_mb, dtype: float64

In [15]:
#save to csv
sub_clean.to_csv('../data/cleaned/clean_subscribers.csv', index=False)
usage_clean.to_csv('../data/cleaned/clean_data_usage.csv', index=False)
pkg_raw.to_csv('../data/cleaned/clean_package.csv', index=False)

## **Section 4. Analytics**

### **4.1 Question 1**

### **4.2 Question 2**

## **Section 5. Conclusion and Recommendation**

**5.1 Conclusion**

**5.2 Recommendation**